In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report

In [4]:
print(torch.__version__)

2.11.0+cu128


In [6]:
texts = [
    "I loved this movie",
    "This movie was amazing",
    "The film was fantastic",
    "I really enjoyed the movie",
    "What a wonderful experience",
    "The acting was excellent",
    "I hated this movie",
    "This movie was terrible",
    "The film was awful",
    "I really disliked the movie",
    "What a horrible experience",
    "The acting was terrible",
]

labels = [
    1, 1, 1, 1, 1, 1,
    0, 0, 0, 0, 0, 0
]

df = pd.DataFrame({
    "text": texts,
    "label": labels
})


In [7]:
df

,text,label
0,I loved this movie,1
1,This movie was amazing,1
2,The film was fantastic,1
3,I really enjoyed the movie,1
4,What a wonderful experience,1
5,The acting was excellent,1
6,I hated this movie,0
7,This movie was terrible,0
8,The film was awful,0
9,I really disliked the movie,0


In [12]:
vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(df["text"])

print("Shape:", X.shape)
# (a, b) a => features/rows b=> unique words

Shape: (12, 20)


In [10]:
print(vectorizer.get_feature_names_out())

['acting' 'amazing' 'awful' 'disliked' 'enjoyed' 'excellent' 'experience'
 'fantastic' 'film' 'hated' 'horrible' 'loved' 'movie' 'really' 'terrible'
 'the' 'this' 'was' 'what' 'wonderful']


"I loved this movie"

        ↓
   TF-IDF Vectorizer

        ↓
[0.0, 0.0, 0.57, ...]

        ↓
      MLP

        ↓
        1

        ↓
    Positive

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    labels,
    test_size=0.25,
    random_state=42,
    stratify=labels
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 9
Testing samples: 3


In [14]:
X_train = torch.tensor(X_train.toarray(), dtype=torch.float32)
X_test = torch.tensor(X_test.toarray(), dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

In [15]:
train_dataset = TensorDataset(X_train, y_train)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True
)

In [16]:
class MLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x):
        return self.network(x)

In [17]:
model = MLP(
    input_size=X_train.shape[1],
    hidden_size=16,
    num_classes=2
)

print(model)

MLP(
  (network): Sequential(
    (0): Linear(in_features=20, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=2, bias=True)
  )
)


              TF-IDF
                │
                │ 20 numbers
                ▼
       ┌─────────────────┐
       │ Linear 20 → 16  │
       └─────────────────┘
                │
                │ 16 numbers
                ▼
            ┌───────┐
            │ ReLU  │
            └───────┘
                │
                │ 16 numbers
                ▼
       ┌─────────────────┐
       │ Linear 16 → 2   │
       └─────────────────┘
                │
                │ 2 logits
                ▼
       ┌─────────────────┐
       │ negative/positive│
       └─────────────────┘

In [18]:
x = X_train[0]

print("Input shape:", x.shape)
print("Input:", x)

Input shape: torch.Size([20])
Input: tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.7492, 0.0000, 0.0000, 0.4224, 0.0000, 0.0000, 0.0000, 0.5102, 0.0000,
        0.0000, 0.0000])


In [19]:
output = model(x)

print("Output shape:", output.shape)
print("Output:", output)

Output shape: torch.Size([2])
Output: tensor([0.2177, 0.0246], grad_fn=<ViewBackward0>)


In [20]:
x = X_train[0]

linear1_output = model.network[0](x)
relu_output = model.network[1](linear1_output)
final_output = model.network[2](relu_output)

print("After Linear:", linear1_output)
print("After ReLU:", relu_output)
print("Final logits:", final_output)

After Linear: tensor([ 0.0130, -0.0942,  0.2799, -0.0356,  0.2723,  0.2642,  0.0406, -0.2775,
         0.1322, -0.1699,  0.0629, -0.0601, -0.0618,  0.0625,  0.0894,  0.0868],
       grad_fn=<ViewBackward0>)
After ReLU: tensor([0.0130, 0.0000, 0.2799, 0.0000, 0.2723, 0.2642, 0.0406, 0.0000, 0.1322,
        0.0000, 0.0629, 0.0000, 0.0000, 0.0625, 0.0894, 0.0868],
       grad_fn=<ReluBackward0>)
Final logits: tensor([0.2177, 0.0246], grad_fn=<ViewBackward0>)


**Learnable Params**

First layer:
20 × 16 + 16 = 336

Second layer:
16 × 2 + 2 = 34

Total:
336 + 34 = 370

In [21]:
criterion = nn.CrossEntropyLoss()

In [22]:
x = X_train[0]
y = y_train[0]

output = model(x)

loss = criterion(output.unsqueeze(0), y.unsqueeze(0))

print("Model output:", output)
print("Actual label:", y)
print("Loss:", loss.item())

Model output: tensor([0.2177, 0.0246], grad_fn=<ViewBackward0>)
Actual label: tensor(0)
Loss: 0.6012495756149292


A simple analogy

Imagine you have 20 books on a shelf:

[book1, book2, ..., book20]

That's one collection.

Now suppose you're organizing books into boxes, where each box contains 20 books.

You could represent one box as:

[1 box, 20 books]

unsqueeze(0) is essentially saying:

"Put this collection inside a batch/container dimension."